In [6]:
%pip install numpy pdfplumber torch faiss-cpu transformers sentence-transformers tqdm flash-attention

This notebook is for testing Retrieval-Augmented Generation (RAG) on SEC 10-K data.

In [2]:
import os
import json
import faiss
import torch
import pdfplumber
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from pathlib import Path

In [3]:
# Confirm we are using a suitable runtime
!nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits

import torch

def check_gpu_memory():
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {gpu_memory:.2f} GB")
        if gpu_memory >= 40:
            print("Sufficient VRAM: At least 40 GB available")
        else:
            print("Insufficient VRAM: Less than 40 GB available")
    else:
        print("No GPU available")

check_gpu_memory()

40960
GPU Memory: 42.47 GB
Sufficient VRAM: At least 40 GB available


In [8]:
# Load the model, tokenizer, and dataset
from google.colab import drive
drive.mount('/content/drive')

import os

# Set up the cache directory
cache_dir = "/content/drive/My Drive/huggingface_cache"
os.makedirs(cache_dir, exist_ok=True)

# Model and device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
retriever_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2').to(device)
rags_model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3.5-mini-instruct',
                                                 cache_dir=cache_dir,
                                                 trust_remote_code=True).to(device)
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3.5-mini-instruct',
                                                 cache_dir=cache_dir,
                                                 trust_remote_code=True)
generator = pipeline('text-generation', model=rags_model, tokenizer=tokenizer, device=0 if device=='cuda' else -1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [46]:
# FAISS index setup
index_path = 'faiss_index.bin'
corpus_path = 'corpus.json'

# Global variables
index = None
corpus = []

# Load FAISS index and corpus if they exist
if os.path.exists(index_path):
    index = faiss.read_index(index_path)
else:
    index = faiss.IndexFlatL2(384)  # Initialize an empty FAISS index

if os.path.exists(corpus_path):
    with open(corpus_path, 'r') as f:
        corpus = json.load(f)

Processing PDFs: 100%|██████████| 1/1 [00:19<00:00, 19.47s/it]


In [67]:
# Load and preprocess PDFs
def extract_text_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = '\n'.join(page.extract_text() or '' for page in pdf.pages)
    return text

# Chunking function
def chunk_text(text, chunk_size=500):
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

# Build FAISS index
def build_index(data_folder):
    global index, corpus
    pdf_files = list(Path(data_folder).rglob('*.pdf'))
    for pdf_file in tqdm(pdf_files, desc='Processing PDFs'):
        text = extract_text_from_pdf(pdf_file)
        chunks = chunk_text(text)
        corpus.extend(chunks)
        embeddings = retriever_model.encode(chunks, convert_to_numpy=True)
        index.add(embeddings)
    faiss.write_index(index, 'faiss_index.bin')
    with open('corpus.json', 'w') as f:
        json.dump(corpus, f)

# Retrieval function
def retrieve_relevant_chunks(query, top_k=5):
    query_embedding = retriever_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    if not corpus or any(i >= len(corpus) for i in indices[0]):
        print("Warning: Corpus is empty or indices are out of range. Returning empty list.")
        return []
    return [corpus[i] for i in indices[0]]

# RAG generation function
def generate_response(query):
    query = query.replace("EGNIVIA", "NVIDIA") # Replace to pull the correct chunks
    relevant_chunks = retrieve_relevant_chunks(query)
    query = query.replace("NVIDIA", "EGNIVIA") # Replace prior to inference
    context = '\n'.join(relevant_chunks)
    context = context.replace("NVIDIA", "EGNIVIA") # Replace context to avoid inteference from existing training data
    input_prompt = tokenizer.apply_chat_template([{"role": "system", "content": f"{context}"}, {"role": "user", "content": f"{query}"}], tokenize=False, add_generation_prompt=True, return_tensors="pt")
    response = generator(input_prompt, max_new_tokens=256, do_sample=True)
    return response[0]['generated_text']


In [ ]:
build_index('/content/drive/My Drive/datasets/temp')

In [29]:
#if __name__ == '__main__':
    # import argparse
    # parser = argparse.ArgumentParser()
    # parser.add_argument('--data_folder', type=str, required=True)
    # parser.add_argument('--query', type=str, required=False, default=None)
    # args = parser.parse_args()

    # if args.query:
    #     print(generate_response(args.query))
    # else:
    #     build_index(args.data_folder)


In [68]:
generate_response("What is EGNIVIA's 2024 revenue?")

"<|system|>\nnet (0.1) (0.4) Income before income tax 15.5 36.9 Income tax expense (benefit) (0.7) 0.7 Net income 16.2 % 36.2 % 41 Table of Contents Revenue Revenue by Reportable Segments Year Ended January 29, January 30, $ % 2023 2022 Change Change ($ in millions) Compute & Networking $ 15,068 $ 11,046 $ 4,022 36 % Graphics 11,906 15,868 (3,962) (25)% Total $ 26,974 $ 26,914 $ 60 — % Compute & Networking - The year-on-year increase was led by growth from hyperscale customers and also reflects purchases made by several CSP partners to support multi-year cloud service agreements for our new EGNIVIA AI cloud service offerings and our research and development activities. The increase was also related to the growth in Automotive. CMP contributed an insignificant amount in fiscal year 2023 compared to $550 million in fiscal year 2022. Graphics - The year-on-year decrease primarily reflects lower sell-in to partners to help reduce channel inventory levels as global macro- economic condition

In [60]:
generate_response("What is EGNIVIA's 2024 target market?")

'<|system|>\n10.1 3/19/2021 10.17+ Fiscal Year 2023 Variable Compensation Plan 8-K 0-23985 10.1 3/9/2022 10.18+ Offer Letter between NVIDIA Corporation and Colette Kress, 8-K 0-23985 10.1 9/16/2013 dated September 13, 2013 10.19+ Offer Letter between NVIDIA Corporation and Tim Teter, 8-K 0-23985 10.1 1/19/2017 dated December 16, 2016 10.20+ Offer Letter between NVIDIA Corporation and Donald 8-K 0-23985 10.1 6/17/2019 Robertson, dated May 21, 2019 10.21 Form of Commercial Paper Dealer Agreement between 8-K 0-23985 10.1 12/15/2017 NVIDIA Corporation, as Issuer, and the Dealer party thereto 21.1* List of Registrant\'s Subsidiaries 23.1* Consent of PricewaterhouseCoopers LLP 24.1* Power of Attorney (included in signature page) 31.1* Certification of Chief Executive Officer as required by Rule 13a-14(a) of the Securities Exchange Act of 1934 31.2* Certification of Chief Financial Officer as required by Rule 13a-14(a) of the Securities Exchange Act of 1934 32.1#* Certification of Chief Execu